In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


In [3]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')
device

'cuda'

In [4]:
torch.manual_seed(42)

In [ ]:
df = pd.read_csv("Datasets/fashion-mnist_train.csv")
df.head(15)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,0,0,0,0,0,0,0,5,0,...,0.0,0.0,0.0,30.0,43.0,0.0,0.0,0.0,0.0,0.0
3,0,0,0,0,1,2,0,0,0,0,...,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,3,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4,0,0,0,5,4,5,5,3,5,...,7.0,8.0,7.0,4.0,3.0,7.0,5.0,0.0,0.0,0.0
6,4,0,0,0,0,0,0,0,0,0,...,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,5,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,4,0,0,0,0,0,0,3,2,0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,8,0,0,0,0,0,0,0,0,0,...,203.0,214.0,166.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
print(f"Number of NaN values before cleaning: {df.isnull().sum().sum()}")
df.fillna(0, inplace=True)
print(f"Number of NaN values after cleaning: {df.isnull().sum().sum()}")

Number of NaN values before cleaning: 213
Number of NaN values after cleaning: 0


In [7]:
X = df.iloc[:, 1:].values
y = df.iloc[:,0].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [9]:
X_train = X_train/255.0
X_test = X_test/255.0

In [10]:
class FMNIST(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):

    return len(self.features)


  def __getitem__(self, index):

    return self.features[index], self.labels[index]

In [11]:
train_data = FMNIST(X_train, y_train)
test_data = FMNIST(X_test, y_test)

In [20]:
class ANN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden, num_neurons, dropout_rate):
    super().__init__()
    layers = []
    for i in range(num_hidden):
      layers.append(nn.Linear(input_dim, num_neurons))
      layers.append(nn.BatchNorm1d(num_neurons))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = num_neurons

    layers.append(nn.Linear(num_neurons, output_dim))

    self.model = nn.Sequential(*layers)


  def forward(self, x):
    return self.model(x)

In [21]:
!pip install optuna

In [22]:
def objective(trial):
  num_hidden = trial.suggest_int('number of hidden layers', 2, 5)
  num_neurons = trial.suggest_int('number of neurons per layer',8,256, step = 8)
  learning_rate = trial.suggest_float("learning rate", 1e-5, 1e-1, log = True)
  epochs = trial.suggest_int('epochs', 10, 50, step = 10)
  dropout_rate = trial.suggest_float('Dropout rate', 0.1, 0.5, step = 0.1)
  batch_size = trial.suggest_categorical('batch_size', [16,32,64,128])
  optimizer = trial.suggest_categorical('optimizer', ['SGD', 'Adam', 'RMSprop'])
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log = True)

  train_loader = DataLoader(train_data, batch_size = 32, shuffle = True, pin_memory=True)
  test_loader = DataLoader(test_data, batch_size = 32, shuffle = False, pin_memory = True)

  input_dim = 784
  output_dim = 10

  model = ANN(input_dim, output_dim, num_hidden, num_neurons, dropout_rate)
  model = model.to(device)


  loss_function = nn.CrossEntropyLoss()
  if optimizer == 'SGD':
      optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = 1e-4)
  elif optimizer == 'Adam':
      optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = 1e-4)
  else:
      optimizer = optim.RMSprop(model.parameters(), lr = learning_rate, weight_decay = 1e-4)



  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      y_pred = model(batch_features)

      loss = loss_function(y_pred, batch_labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  model.eval()
  total = 0
  correct = 0
  with torch.no_grad():
    for batch_features, batch_labels in test_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        outputs = model(batch_features)
        _, output_labels = torch.max(outputs, 1)
        total += batch_labels.shape[0]
        correct = correct + (output_labels == batch_labels).sum().item()


  accuracy = correct/total


  return accuracy



In [23]:
import optuna
study = optuna.create_study(direction = 'maximize')
study.optimize(objective, n_trials = 50)

[I 2026-07-08 09:56:18,525] A new study created in memory with name: no-name-50170557-4984-4690-b608-5ea4befe024c
[I 2026-07-08 09:58:05,557] Trial 0 finished with value: 0.6969005285920231 and parameters: {'number of hidden layers': 2, 'number of neurons per layer': 192, 'learning rate': 0.05028212937670135, 'epochs': 40, 'Dropout rate': 0.5, 'batch_size': 128, 'optimizer': 'RMSprop', 'weight_decay': 0.0006140862623465331}. Best is trial 0 with value: 0.6969005285920231.
[I 2026-07-08 09:58:31,763] Trial 1 finished with value: 0.8653291686689092 and parameters: {'number of hidden layers': 3, 'number of neurons per layer': 216, 'learning rate': 0.0909196790377556, 'epochs': 10, 'Dropout rate': 0.1, 'batch_size': 16, 'optimizer': 'SGD', 'weight_decay': 0.0005495902773078712}. Best is trial 1 with value: 0.8653291686689092.
[I 2026-07-08 09:59:42,697] Trial 2 finished with value: 0.735223450264296 and parameters: {'number of hidden layers': 2, 'number of neurons per layer': 8, 'learning 

In [26]:
print(study.best_value)
print(study.best_params)

0.8916386352715041
{'number of hidden layers': 3, 'number of neurons per layer': 240, 'learning rate': 0.006603774004846118, 'epochs': 40, 'Dropout rate': 0.1, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.0009952705754150337}


Best parameters found:


*   hidden layers:3
*   number of neurons:240
*   learning rate: 0.006603
*   epochs:40
*   Droupout rate: 0.1
*   batch size: 64
*   optimizer: SGD
*   weight decay: 0.0009952






